# Demonstrasi: Pembersihan Data & Log Keputusan — PIF304 Pengantar Data Sains
**Pertemuan 5 — Sub-CPMK S03: Memilih prosedur pembersihan data**

Membersihkan data **bukan** soal menjalankan `dropna()`. Setiap tindakan adalah **keputusan** yang mengubah hasil analisis, sehingga harus (1) beralasan, (2) tercatat, dan (3) dapat diulang.

Alur kerja yang kita pakai: **Periksa → Diagnosis → Putuskan → Terapkan → Catat → Verifikasi**

**Dataset:** `Data_Survei_Latihan_Kotor_P5.xlsx` — versi latihan dari survei kelas. Sebagian masalah (duplikasi, salah ketik, format tidak konsisten, nilai kosong) **sengaja ditambahkan** untuk latihan; nilai `jarak_km` 500/1000 adalah isian asli.

Demo ini menangani: **duplikasi, `usia`, `perangkat_utama`, `jarak_km`, `waktu_tempuh_menit`**. Kolom lainnya menjadi tugas Anda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
# Palet grafik (sudah diuji aman untuk buta warna): satu seri = TEAL; pembanding = ORANGE, VIOLET; latar = GREY
NAVY, TEAL, ORANGE, VIOLET, GREY = "#13233F", "#1293A6", "#E4683A", "#6C63C9", "#B8BDC4"
PALET = [TEAL, ORANGE, VIOLET]

### Unggah File Data
Jalankan sel di bawah, klik **Choose Files**, lalu pilih **`Data_Survei_Latihan_Kotor_P5.xlsx`** yang sudah Anda unduh dari LMS.

In [ ]:
try:
    from google.colab import files
    print("Silakan pilih file: Data_Survei_Latihan_Kotor_P5.xlsx")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih file-nya.")
    nama_file = list(uploaded.keys())[0]
except ImportError:
    # Di luar Google Colab (mis. Jupyter lokal): letakkan file di folder yang sama dengan notebook
    nama_file = "Data_Survei_Latihan_Kotor_P5.xlsx"

mentah = pd.read_excel(nama_file)   # sheet pertama = "Data"
print("File:", nama_file, "| ukuran:", mentah.shape)
mentah.head()

## 0. Dua Aturan Emas
1. **Data mentah tidak pernah ditimpa.** Kita bekerja pada salinan.
2. **Setiap perubahan dicatat** dalam log keputusan.

In [ ]:
df = mentah.copy()          # 'mentah' tidak disentuh lagi
log = []                    # log keputusan pembersihan

def catat(kolom, masalah, bukti, keputusan, alasan, baris_terdampak):
    log.append({"kolom": kolom, "masalah": masalah, "bukti": bukti, "keputusan": keputusan,
                "alasan": alasan, "baris_terdampak": baris_terdampak})

print("Ukuran awal:", df.shape)

## 1. Periksa: Gambaran Umum

In [ ]:
df.info()

In [ ]:
ringkas = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_kosong": df.isna().sum(), "n_unik": df.nunique()})
ringkas

**Yang perlu dicurigai dari tabel di atas:**
- `usia` bertipe teks/`object`, padahal seharusnya angka.
- `perangkat_utama`, `moda_transportasi`, `semester` punya **lebih banyak nilai unik** daripada yang tercatat di data dictionary (3, 3, dan 2).
- Ada nilai kosong di beberapa kolom.
- Jumlah baris 48, padahal respondennya 45.

## 2. Duplikasi

In [ ]:
dup = df[df.duplicated(keep=False)].sort_values("nama")
print("Baris yang terlibat duplikasi:", len(dup))
dup[["nama", "usia", "semester", "moda_transportasi", "jarak_km"]]

**Diagnosis:** tiga responden muncul dua kali dengan isi **identik di semua kolom** (termasuk pengenal `nama`). Ini pola pengiriman formulir ganda, bukan dua orang berbeda.

**Keputusan:** simpan satu, hapus salinannya.

> Hati-hati: jika hanya *sebagian* kolom yang sama, itu bisa saja dua orang berbeda dengan jawaban mirip. Duplikasi diputuskan berdasarkan **kunci** (pengenal), bukan kemiripan.

In [ ]:
sebelum = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
catat("(semua)", "Duplikasi persis", "3 pengenal muncul 2x dengan isi identik", "Hapus salinan, simpan 1",
      "Pengiriman formulir ganda; pengenal sama", sebelum - len(df))
print(f"{sebelum} -> {len(df)} baris")

## 3. Tipe Data dan Rentang: `usia`

In [ ]:
bukan_angka = df[pd.to_numeric(df["usia"], errors="coerce").isna()]
bukan_angka[["nama", "usia"]]

**Diagnosis:** dua isian berbentuk teks ("19 tahun", "20 thn"). Maknanya jelas, hanya formatnya yang salah → aman untuk **diperbaiki**: ambil angkanya.

In [ ]:
usia_baru = pd.to_numeric(df["usia"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
catat("usia", "Tipe salah (teks)", "'19 tahun', '20 thn'", "Ambil bagian angka, ubah ke numerik",
      "Makna isian jelas; hanya format", int((df["usia"].astype(str) != usia_baru.astype("Int64").astype(str)).sum()))
df["usia"] = usia_baru
df["usia"].describe()

In [ ]:
# Validasi rentang dari data dictionary: 16-60 tahun
df.loc[~df["usia"].between(16, 60), ["nama", "usia", "semester"]]

**Diagnosis:** usia 91 untuk mahasiswa Semester 3 hampir pasti salah ketik (mungkin 19 terbalik). Tetapi "mungkin" **bukan** bukti.

| Pilihan | Risiko |
|---|---|
| Ubah menjadi 19 | Kita mengarang data berdasarkan tebakan |
| Hapus barisnya | Kehilangan 11 jawaban lain yang sah |
| **Jadikan kosong + beri tanda** | Jujur: nilainya tidak diketahui |

**Keputusan:** jadikan `NaN` dan beri penanda.

In [ ]:
tidak_wajar = ~df["usia"].between(16, 60)
df["usia_tidak_valid"] = tidak_wajar
df.loc[tidak_wajar, "usia"] = np.nan
catat("usia", "Di luar rentang 16-60", "usia = 91 (Semester 3)", "Jadikan NaN + kolom penanda usia_tidak_valid",
      "Nilai benar tidak dapat dipastikan; baris tetap dipakai untuk kolom lain", int(tidak_wajar.sum()))
df["usia"].describe().round(2)

## 4. Inkonsistensi Kategori: `perangkat_utama`

In [ ]:
df["perangkat_utama"].value_counts(dropna=False)

**Diagnosis:** variasi kapitalisasi, spasi di akhir, dan sinonim ("HP"). Komputer menganggapnya kategori berbeda, sehingga proporsi menjadi salah.

**Keputusan:** (1) rapikan spasi dan huruf, (2) petakan sinonim dengan **kamus pemetaan yang eksplisit** agar bisa diperiksa orang lain.

In [ ]:
peta_perangkat = {"smartphone": "Smartphone", "hp": "Smartphone", "laptop": "Laptop", "lainnya": "Lainnya"}
kunci = df["perangkat_utama"].str.strip().str.lower()
belum_terpetakan = sorted(set(kunci.dropna()) - set(peta_perangkat))
print("Nilai yang belum ada di kamus pemetaan:", belum_terpetakan)

berubah = int((df["perangkat_utama"] != kunci.map(peta_perangkat)).sum())
df["perangkat_utama"] = kunci.map(peta_perangkat)
catat("perangkat_utama", "Penulisan kategori tidak konsisten", "smartphone/SMARTPHONE/HP/hp/'Smartphone '/laptop",
      "strip + lower + pemetaan eksplisit ke 3 kategori baku", "Mengacu nilai valid di data dictionary", berubah)
df["perangkat_utama"].value_counts(dropna=False)

## 5. Nilai Ekstrem: `jarak_km`

In [ ]:
q1, q3 = df["jarak_km"].quantile([0.25, 0.75])
iqr = q3 - q1
batas_atas = q3 + 1.5 * iqr
print(f"Q1 = {q1}, Q3 = {q3}, IQR = {iqr:.2f} -> batas atas aturan 1,5 x IQR = {batas_atas:.2f} km")
df.loc[df["jarak_km"] > batas_atas, ["nama", "moda_transportasi", "jarak_km", "waktu_tempuh_menit"]].sort_values("jarak_km")

Aturan IQR menandai **dua jenis** baris yang sangat berbeda:

| Baris | Diagnosis | Dasar |
|---|---|---|
| 9–10 km, sepeda motor, 22–25 menit | **Sah tetapi jarang** | Konsisten dengan moda dan waktu tempuh |
| 500/1000, jalan kaki, 8–15 menit | **Salah satuan** (meter) | Mustahil secara fisik; 500 m dalam 10 menit masuk akal |

Aturan statistik hanya **menandai kandidat**. Diagnosis membutuhkan pengetahuan konteks.

**Keputusan:** nilai sah dibiarkan. Untuk 500/1000, kita buat kolom baru `jarak_km_bersih` (dibagi 1000) dan penanda `jarak_dikoreksi`; kolom asli **tetap disimpan**.

In [ ]:
dugaan_meter = (df["jarak_km"] >= 100) & (df["moda_transportasi"].str.strip().str.lower() == "jalan kaki")
df["jarak_km_bersih"] = np.where(dugaan_meter, df["jarak_km"] / 1000, df["jarak_km"])
df["jarak_dikoreksi"] = dugaan_meter
catat("jarak_km", "Nilai ekstrem (diduga salah satuan)", "500/1000 km, jalan kaki, 8-15 menit",
      "Kolom baru jarak_km_bersih = nilai/1000 + penanda jarak_dikoreksi", "Konsisten sebagai meter; kolom asli disimpan (asumsi belum dikonfirmasi)",
      int(dugaan_meter.sum()))

print("Sebelum :", df["jarak_km"].describe()[["mean", "50%", "max"]].round(2).to_dict())
print("Sesudah :", df["jarak_km_bersih"].describe()[["mean", "50%", "max"]].round(2).to_dict())

## 6. Nilai Kosong: `waktu_tempuh_menit`

In [ ]:
df.loc[df["waktu_tempuh_menit"].isna(), ["nama", "moda_transportasi", "jarak_km_bersih", "waktu_tempuh_menit"]]

Tiga pilihan utama, masing-masing dengan konsekuensi:

| Strategi | Kapan cocok | Risiko |
|---|---|---|
| **Hapus baris** | Kosong sedikit dan acak; kolom itu inti analisis | Sampel mengecil; bias jika kosongnya tidak acak |
| **Imputasi** (median/modus, per kelompok) | Kosong sedikit; butuh jumlah baris utuh | Sebaran menyempit; nilai "karangan" |
| **Biarkan + tandai** | Analisis dapat mengabaikan NaN | Jumlah n berbeda antaranalisis |

Mari bandingkan dampaknya terhadap ringkasan.

In [ ]:
w = df["waktu_tempuh_menit"]
moda_rapi = df["moda_transportasi"].str.strip().str.lower()
opsi = {
    "Biarkan NaN":             w,
    "Hapus baris":             w.dropna(),
    "Imputasi median global":  w.fillna(w.median()),
    "Imputasi median per moda": w.fillna(w.groupby(moda_rapi).transform("median")),
}
pd.DataFrame({k: {"n": v.count(), "mean": round(v.mean(), 2), "median": v.median(), "simpangan_baku": round(v.std(), 2)} for k, v in opsi.items()}).T

**Keputusan untuk kelas ini:** imputasi **median per moda transportasi**, karena waktu tempuh jelas bergantung pada moda, dan hanya 3 dari 45 nilai yang kosong. Kita tambahkan penanda agar analisis berikutnya dapat mengecualikan nilai hasil imputasi bila perlu.

> Tidak ada strategi yang "selalu benar". Yang dinilai adalah **alasan** dan **keterlacakan** keputusan.

In [ ]:
kosong = df["waktu_tempuh_menit"].isna()
df["waktu_tempuh_diimputasi"] = kosong
df["waktu_tempuh_menit"] = w.fillna(w.groupby(moda_rapi).transform("median"))
catat("waktu_tempuh_menit", "Nilai kosong", "3 dari 45 baris", "Imputasi median per moda + penanda waktu_tempuh_diimputasi",
      "Waktu tempuh bergantung moda; kosong sedikit", int(kosong.sum()))
df.loc[kosong, ["nama", "moda_transportasi", "waktu_tempuh_menit", "waktu_tempuh_diimputasi"]]

## 7. Verifikasi
Jalankan ulang aturan validasi dari data dictionary pada kolom yang sudah dibersihkan.

In [ ]:
aturan = {
    "tidak ada duplikasi":       not df.duplicated().any(),
    "usia numerik":              pd.api.types.is_numeric_dtype(df["usia"]),
    "usia 16-60 atau kosong":    bool((df["usia"].between(16, 60) | df["usia"].isna()).all()),
    "perangkat_utama 3 kategori": set(df["perangkat_utama"].dropna()) <= {"Smartphone", "Laptop", "Lainnya"},
    "jarak_km_bersih 0-50":      bool(df["jarak_km_bersih"].between(0, 50).all()),
    "waktu_tempuh tanpa kosong": bool(df["waktu_tempuh_menit"].notna().all()),
    "jumlah baris = 45":         len(df) == 45,
}
pd.Series(aturan, name="lolos")

## 8. Log Keputusan Pembersihan
Inilah artefak terpenting pertemuan ini.

In [ ]:
log_df = pd.DataFrame(log)
log_df

In [ ]:
log_df.to_csv("log_pembersihan_demo_P5.csv", index=False)
df.to_csv("survei_setengah_bersih_demo_P5.csv", index=False)
print("Tersimpan: log_pembersihan_demo_P5.csv dan survei_setengah_bersih_demo_P5.csv")
print("Kolom yang BELUM dibersihkan (tugas Anda): semester, moda_transportasi, penggunaan_internet, platform_belajar, tingkat_ketertarikan")

## Ringkasan
| Masalah | Cara mendeteksi | Pilihan tindakan | Yang wajib dicatat |
|---|---|---|---|
| Duplikasi | `duplicated()` pada kunci | hapus salinan | dasar menyebutnya duplikat |
| Tipe salah | `dtypes`, `to_numeric(errors="coerce")` | konversi | pola yang diperbaiki |
| Di luar rentang | aturan dari data dictionary | koreksi / kosongkan / hapus | mengapa tidak menebak |
| Kategori tidak konsisten | `value_counts()` | strip, lower, pemetaan | kamus pemetaan |
| Nilai ekstrem | IQR + pemeriksaan konteks | biarkan / koreksi / tandai | diagnosis tiap kelompok |
| Nilai kosong | `isna().sum()` | hapus / imputasi / tandai | strategi & dampaknya |

**Etika:** membersihkan data untuk memperoleh hasil yang "lebih bagus" adalah manipulasi. Putuskan aturan pembersihan **sebelum** melihat dampaknya pada kesimpulan, dan laporkan apa adanya.